# Bug fix: `product_category_name_english` en `tad_ventas`

## Problema

Durante la auditoría del proyecto se detectó que la columna `product_category_name_english` en `tad_ventas` está **100 % nula** (tipo `float64`, 0 valores únicos):

```
13  tad_ventas  product_category_name_english   100.00 %   0 valores únicos
```

## Causa raíz

En la limpieza de `products` se aplicó un `str.title()` y se reemplazaron los `_` por espacios:

```python
products_clean['product_category_name'] = (
    products_clean['product_category_name']
    .str.replace('_', ' ').str.title()
)
```

lo que convierte `esporte_lazer` → `Esporte Lazer`. Después, en el ETL, el merge se hace contra `category_translation` que **conserva el formato `snake_case` original** (`esporte_lazer`):

```python
products = products.merge(category_translation, on="product_category_name", how="left")
```

Como las llaves nunca coinciden, el merge devuelve NaN para todas las filas. El bug pasó desapercibido porque el ETL no validó la cobertura del merge.

## Solución

Re-poblar `product_category_name_english` directamente sobre `tad_ventas` usando un mapeo canónico, sin tocar el resto del pipeline. Antes de ejecutar este notebook ya se respaldó el archivo en `tad_ventas_backup_pre_bugfix.csv`.

In [1]:
import os

import numpy as np
import pandas as pd

CSV_DIR = os.environ.get('CSV_DIR', '../CSV')

tad_ventas = pd.read_csv(os.path.join(CSV_DIR, '02_tad_ventas.csv'))
print('Filas:', len(tad_ventas))
print('Nulos en product_category_name_english (antes):',
      tad_ventas['product_category_name_english'].isna().sum())


Filas: 112650
Nulos en product_category_name_english (antes): 0


## Diccionario canónico de traducción

Reconstruido a partir del archivo oficial `product_category_name_translation.csv` de Olist (71 entradas), con las erratas ortográficas corregidas en el notebook de limpieza (`costruction` → `construction`, `confort` → `comfort`, `craftmanship` → `craftsmanship`, `fashio_` → `fashion_`) y se incluye `sin_categoria` → `no_category` para los registros que el equipo etiquetó como faltantes.

In [2]:
TRANSLATION = {
    'agro_industria_e_comercio': 'agro_industry_and_commerce',
    'alimentos': 'food',
    'alimentos_bebidas': 'food_drink',
    'artes': 'art',
    'artes_e_artesanato': 'arts_and_craftsmanship',
    'artigos_de_festas': 'party_supplies',
    'artigos_de_natal': 'christmas_supplies',
    'audio': 'audio',
    'automotivo': 'auto',
    'bebes': 'baby',
    'bebidas': 'drinks',
    'beleza_saude': 'health_beauty',
    'brinquedos': 'toys',
    'cama_mesa_banho': 'bed_bath_table',
    'casa_conforto': 'home_comfort',
    'casa_conforto_2': 'home_comfort_2',
    'casa_construcao': 'home_construction',
    'cds_dvds_musicais': 'cds_dvds_musicals',
    'cine_foto': 'cine_photo',
    'climatizacao': 'air_conditioning',
    'consoles_games': 'consoles_games',
    'construcao_ferramentas_construcao': 'construction_tools_construction',
    'construcao_ferramentas_ferramentas': 'construction_tools_tools',
    'construcao_ferramentas_iluminacao': 'construction_tools_lights',
    'construcao_ferramentas_jardim': 'construction_tools_garden',
    'construcao_ferramentas_seguranca': 'construction_tools_safety',
    'cool_stuff': 'cool_stuff',
    'dvds_blu_ray': 'dvds_blu_ray',
    'eletrodomesticos': 'home_appliances',
    'eletrodomesticos_2': 'home_appliances_2',
    'eletronicos': 'electronics',
    'eletroportateis': 'small_appliances',
    'esporte_lazer': 'sports_leisure',
    'fashion_bolsas_e_acessorios': 'fashion_bags_accessories',
    'fashion_calcados': 'fashion_shoes',
    'fashion_esporte': 'fashion_sport',
    'fashion_roupa_feminina': 'fashion_female_clothing',
    'fashion_roupa_infanto_juvenil': 'fashion_childrens_clothes',
    'fashion_roupa_masculina': 'fashion_male_clothing',
    'fashion_underwear_e_moda_praia': 'fashion_underwear_beach',
    'ferramentas_jardim': 'garden_tools',
    'flores': 'flowers',
    'fraldas_higiene': 'diapers_and_hygiene',
    'industria_comercio_e_negocios': 'industry_commerce_and_business',
    'informatica_acessorios': 'computers_accessories',
    'instrumentos_musicais': 'musical_instruments',
    'la_cuisine': 'la_cuisine',
    'livros_importados': 'books_imported',
    'livros_interesse_geral': 'books_general_interest',
    'livros_tecnicos': 'books_technical',
    'malas_acessorios': 'luggage_accessories',
    'market_place': 'market_place',
    'moveis_colchao_e_estofado': 'furniture_mattress_and_upholstery',
    'moveis_cozinha_area_de_servico_jantar_e_jardim': 'kitchen_dining_laundry_garden_furniture',
    'moveis_decoracao': 'furniture_decor',
    'moveis_escritorio': 'office_furniture',
    'moveis_quarto': 'furniture_bedroom',
    'moveis_sala': 'furniture_living_room',
    'musica': 'music',
    'papelaria': 'stationery',
    'pc_gamer': 'pc_gamer',
    'pcs': 'computers',
    'perfumaria': 'perfumery',
    'pet_shop': 'pet_shop',
    'portateis_casa_forno_e_cafe': 'small_appliances_home_oven_and_coffee',
    'portateis_cozinha_e_preparadores_de_alimentos': 'small_appliances_kitchen_food_preparation',
    'relogios_presentes': 'watches_gifts',
    'seguros_e_servicos': 'security_and_services',
    'sinalizacao_e_seguranca': 'signaling_and_security',
    'tablets_impressao_imagem': 'tablets_printing_image',
    'telefonia': 'telephony',
    'telefonia_fixa': 'fixed_telephony',
    'utilidades_domesticas': 'housewares',
    'sin_categoria': 'no_category',
}

print('Categorías mapeadas:', len(TRANSLATION))

Categorías mapeadas: 74


## Aplicación del fix

Convertimos el nombre Title Case a `snake_case` y aplicamos el mapeo. Si quedara alguna categoría sin match (porque el dataset trae un valor no estándar), se rellena con `'unknown'` y se reporta para auditoría.

In [3]:
def title_to_snake(s):
    if pd.isna(s):
        return s
    return str(s).strip().lower().replace(' ', '_')

tad_ventas['_cat_snake'] = tad_ventas['product_category_name'].apply(title_to_snake)
tad_ventas['product_category_name_english'] = tad_ventas['_cat_snake'].map(TRANSLATION)

# Auditoría
sin_match = tad_ventas[
    tad_ventas['product_category_name_english'].isna() &
    tad_ventas['_cat_snake'].notna()
]['_cat_snake'].value_counts()

print('Categorías sin match:', len(sin_match))
if len(sin_match) > 0:
    print(sin_match)
    tad_ventas['product_category_name_english'] = (
        tad_ventas['product_category_name_english'].fillna('unknown')
    )

# Limpieza de columna auxiliar
tad_ventas = tad_ventas.drop(columns=['_cat_snake'])

print('\nNulos en product_category_name_english (después):',
      tad_ventas['product_category_name_english'].isna().sum())
print('Valores únicos:', tad_ventas['product_category_name_english'].nunique())
print('Top categorías traducidas:')
print(tad_ventas['product_category_name_english'].value_counts().head(10))

Categorías sin match: 0

Nulos en product_category_name_english (después): 0
Valores únicos: 74
Top categorías traducidas:
product_category_name_english
bed_bath_table           11115
health_beauty             9670
sports_leisure            8641
furniture_decor           8334
computers_accessories     7827
housewares                6964
watches_gifts             5991
telephony                 4545
garden_tools              4347
auto                      4235
Name: count, dtype: int64


## Validación

Confirmamos que la cobertura del merge es 100 %, que el número de filas no cambió y que las relaciones portugués ↔ inglés son consistentes (cada `product_category_name` corresponde a una única traducción).

In [4]:
# Asserts de calidad
assert len(tad_ventas) == 112650, 'El número de filas cambió'
assert tad_ventas['product_category_name_english'].notna().all(), 'Quedan nulos en la traducción'

consistencia = (
    tad_ventas
    .groupby('product_category_name')['product_category_name_english']
    .nunique()
)
assert (consistencia == 1).all(), 'Hay categorías PT con múltiples traducciones EN'

print('OK · Todas las validaciones pasaron.')

OK · Todas las validaciones pasaron.


## Persistencia

Sobrescribimos `tad_ventas.csv`. El backup queda en `tad_ventas_backup_pre_bugfix.csv` por si fuera necesario revertir.

In [5]:
tad_ventas.to_csv(os.path.join(CSV_DIR, '02_tad_ventas.csv'), index=False)
print('tad_ventas.csv reescrito con la columna corregida.')
print('Filas:', len(tad_ventas), '· Columnas:', tad_ventas.shape[1])

tad_ventas.csv reescrito con la columna corregida.
Filas: 112650 · Columnas: 55


## Recarga a BigQuery (opcional)

Si la sábana en BigQuery debe quedar consistente, ejecutar este bloque en Colab con las credenciales de servicio. Misma lógica que el ETL original.

In [6]:
# from google.cloud import bigquery
# client = bigquery.Client(project='mineria-datos-493000')
# job = client.load_table_from_dataframe(
#     tad_ventas,
#     'mineria-datos-493000.smart_supply_chain.tad_ventas',
#     job_config=bigquery.LoadJobConfig(
#         write_disposition='WRITE_TRUNCATE', autodetect=True
#     )
# )
# job.result()
# print('Recarga completada.')